# 강의 06 · 실습 12 — 패턴 6 평가자-최적화자 · (1) 강사 시연

## 1. 문제상황

- 총무팀은 회의가 끝나면 회의록을 한 문장으로 요약해 팀장에게 보냅니다.
- 담당자는 요약을 에이전트 하나에 맡기고 결과를 그대로 보냈습니다.
- 요약에는 결정된 것은 있는데 담당자가 빠져 있거나, 담당자는 있는데 결정된 것이 빠져 있는 경우가 생깁니다.
- 빠진 것을 사람이 확인하고 다시 시키는 일이 회의록마다 반복됩니다.

## 2. 문제와 목표

- **문제**: 요약을 만드는 일과 요약이 기준을 지켰는지 판정하는 일을 한 호출이 같이 하므로, 빠진 것을 잡아내는 단계가 없습니다.
- **목표**: 회의록을 입력하면 generator 노드가 요약을 만들고, evaluator 노드가 결정된 것과 담당자가 둘 다 있는지 구조화 출력으로 판정하고, 반려면 고칠 점을 실어 generator로 되돌리고, 합격이거나 시도 상한에 닿으면 끝나는 처리 흐름을 만듭니다.
    - generator 노드: 회의록 요약을 쓰고, 지적이 있으면 반영해 다시 씁니다.
    - evaluator 노드: 결정된 것과 담당자가 둘 다 있는지를 구조화 출력 `Verdict`(합격/반려와 고칠 점)로 판정합니다.
    - 시도 상한: 3회.
- **목표 달성 여부의 판정 기준**: 회의록을 입력했을 때, 첫 요약이 반려되어 지적이 상태에 남고, 두 번째 요약이 지적을 반영해 합격으로 끝나는 것을 실행 결과에서 확인합니다. 최종 상태의 시도 횟수는 2입니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex12_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태와 평가 규격을 정의합니다.**
    - 회의록(`minutes`), 요약(`summary`), 지적(`feedback`), 판정(`grade`), 시도 횟수(`tries`) 키 다섯 개를 가지는 상태를 선언합니다.
    - 평가 규격 `Verdict`는 `grade` 값을 합격·반려 둘로 제한하고 `feedback`에 고칠 점 한 문장을 담습니다.
    - 시도 상한은 3으로 둡니다.
2. **generator 노드를 만듭니다.**
    - 지적이 비어 있으면 「회의록을 25자 이내 한 문장으로 아주 짧게 요약한다」는 지침으로 회의록만 넣어 첫 요약을 만듭니다.
    - 지적이 있으면 「회의록을 한 문장으로 요약한다. 지난 지적을 반영한다」는 지침으로 지적과 회의록을 함께 넣어 다시 씁니다.
    - 결과를 `summary`에 쓰고 `tries`를 1 올립니다.
3. **evaluator 노드를 만듭니다.**
    - `Verdict` 규격을 건 모델에 「요약이 결정된 것과 담당자를 둘 다 담았으면 합격, 아니면 반려」라는 지침으로 회의록과 요약을 넣어 판정을 받고, `grade`·`feedback` 키에 씁니다.
    - evaluator는 요약을 고치지 않습니다.
4. **그래프에 노드를 등록합니다.**
    - generator·evaluator 두 노드를 이름과 함께 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 generator로, generator에서 evaluator로 가는 고정 엣지를 추가합니다.
    - evaluator 뒤에는 판단 함수 route_summary가 합격이거나 `tries`가 상한이면 END를, 아니면 generator를 돌려주는 조건부 엣지를 추가합니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 회의록과 빈 요약, 빈 지적, 빈 판정, 시도 횟수 0을 넣어 실행한 뒤, 최종 판정과 시도 횟수와 요약을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키와 평가 규격을 선언합니다 | `class SummaryState(TypedDict)`, `class Verdict(BaseModel)` | 1 |
| ② 노드 함수 정의 | 요약을 만드는 generator와 판정만 하는 evaluator를 만듭니다 | `def generator(state) -> dict`, `llm.with_structured_output(Verdict)` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 두 노드를 등록합니다 | `StateGraph(SummaryState)`, `add_node` | 4 |
| ④ 엣지 연결 | 직렬 순서와 되돌림 조건을 정합니다 | `add_edge`, `add_conditional_edges` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 회의록을 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 두 노드가 함께 읽고 쓰는 키를 선언합니다. `feedback`은 evaluator가 쓰고 generator가 읽는 키며, 반려 사유가 이 키를 통해 되돌아갑니다. `tries`는 시도 횟수를 담는 키며 종료를 판정할 때 읽습니다. 평가 규격 `Verdict`가 있으므로 판정은 답변 문장을 뒤져 찾아내지 않고 규격대로 받습니다.

In [ ]:
class Verdict(BaseModel):
    grade: Literal["합격", "반려"] = Field(description="요약이 규격을 지켰는가")
    feedback: str = Field(description="반려라면 무엇을 고칠지 한 문장, 합격이면 빈 문자열")


class SummaryState(TypedDict):
    minutes: str    # 회의록 본문
    summary: str    # 생성된 한 줄 요약
    feedback: str   # 평가자의 반려 사유
    grade: str      # 합격 / 반려
    tries: int      # 시도 횟수


MAX_TRIES = 3

print("상태의 키:", list(SummaryState.__annotations__))
print("판정 값의 범위:", Verdict.model_json_schema()["properties"]["grade"]["enum"])

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- generator는 한 노드가 첫 시도와 다시 쓰기를 함께 맡습니다. 지적이 비어 있으면 첫 프롬프트로, 들어 있으면 지적을 실은 프롬프트로 요약을 만듭니다. 지침은 `SystemMessage`, 회의록과 지적은 `HumanMessage`로 층을 나눕니다.
- 첫 시도는 일부러 아주 짧게 시킵니다. 반려 경로가 실행 결과에 보이도록 만든 규칙입니다.
- evaluator는 회의록 본문과 요약을 함께 넣어 판정을 받습니다. 평가자는 요약을 고치지 않고 판정과 사유만 돌려줍니다.
- 두 노드가 쓰는 키가 나뉘어 있어 서로의 값을 덮지 않습니다.

In [ ]:
grader = llm.with_structured_output(Verdict)


def generator(state: SummaryState) -> dict:
    """요약을 만든다. 지적이 있으면 반영해 다시 쓴다."""
    n = state["tries"] + 1
    if state["feedback"]:
        guide = "회의록을 한 문장으로 요약한다. 지난 지적을 반영한다."
        human = f"지적: {state['feedback']}\n\n{state['minutes']}"
    else:
        guide = "회의록을 25자 이내 한 문장으로 아주 짧게 요약한다."   # 첫 시도는 일부러 짧게
        human = state["minutes"]
    res = llm.invoke([SystemMessage(guide), HumanMessage(human)])
    print(f"  [generator] 진입 시도 {n}회차")
    print(f"    [요약 {n}회차] {res.content.strip()}")
    return {"summary": res.content.strip(), "tries": n}


def evaluator(state: SummaryState) -> dict:
    """요약을 고치지 않고 판정과 사유만 돌려준다."""
    v = grader.invoke([
        SystemMessage("요약이 '결정된 것'과 '담당자'를 둘 다 담았으면 합격, 아니면 반려."),
        HumanMessage(f"회의록: {state['minutes']}\n요약: {state['summary']}"),
    ])
    print(f"  [evaluator] 진입 -> grade={v.grade!r}")
    if v.feedback:
        print(f"    [판정 사유] {v.feedback}")
    return {"grade": v.grade, "feedback": v.feedback}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 생성 노드와 평가 노드 둘뿐입니다.

In [ ]:
g = StateGraph(SummaryState)
g.add_node("generator", generator)
g.add_node("evaluator", evaluator)

print("등록한 노드:", list(g.nodes))

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `route_summary`가 종료 조건 자체입니다. 합격이거나 시도 상한에 닿으면 END, 아니면 생성 노드로 되돌립니다. 상한이 없으면 반려가 이어질 때 루프가 스스로 멈추지 못합니다.

In [ ]:
def route_summary(state: SummaryState) -> str:
    """종료 조건: 합격이거나 상한에 닿으면 END, 아니면 generator로 되돌린다."""
    if state["grade"] == "합격" or state["tries"] >= MAX_TRIES:
        return END
    return "generator"


g.add_edge(START, "generator")
g.add_edge("generator", "evaluator")
g.add_conditional_edges("evaluator", route_summary, ["generator", END])

print("고정 엣지 수:", len(g.edges))

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 회의록과 빈 값들을 넣으면 최종 상태가 돌아옵니다. 실행 중 두 노드가 출력하는 진입 줄로 되돌림 횟수를 봅니다. 아래에서는 회의실 예약 시스템 이전 회의록을 넣습니다.

In [ ]:
graph = g.compile()

MINUTES = ("회의실 예약 시스템을 다음 달부터 사내 포털로 옮기기로 했다. "
           "이전 작업은 총무팀이 맡고, 사용법 안내는 각 부서 총무 담당이 전달한다.")

print(f"=== 회의록: {MINUTES[:30]}... ===")
out = graph.invoke({"minutes": MINUTES, "summary": "", "feedback": "", "grade": "", "tries": 0})

print()
print(f"[최종 상태] grade={out['grade']!r} tries={out['tries']}")
print("--- 요약 ---")
print(out["summary"])
if out["grade"] != "합격":
    print("[마지막 반려 사유]", out["feedback"])

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. `[generator] 진입 시도 1회차` 뒤에 `[evaluator] 진입 -> grade='반려'`와 판정 사유가 출력됩니다. 짧은 첫 요약에서 담당자가 빠졌습니다.
2. `[generator] 진입 시도 2회차`의 요약은 지적을 반영해 결정된 것과 담당자를 둘 다 담고, 그 뒤 `grade='합격'`이 출력됩니다.
3. 최종 상태의 `tries`는 2이고 `grade`는 합격입니다. 세 번째 시도는 일어나지 않았습니다.

반려 사유가 상태의 `feedback` 키를 거쳐 generator로 되돌아간 것이 평가자-최적화자가 동작한 증거입니다.